In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("..")

import numpy as np
from numpy.linalg import norm
from datetime import datetime
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import plotly.graph_objects as go
from tqdm import tqdm
from dataclasses import dataclass
from lunanav.constants import *  # noqa: F403
from lunanav.sim.simulator import SimParams, RigidBody, run_sim, SimResults, reverse_sim_results
from lunanav.sim.sensors import (
    SensorEnvironment, Sensor, SensorSuite, SensorName,
    accelerometer_sensor, gyroscope_sensor,
    laser_altimeter_sensor, laser_velocity_sensor,
    star_tracker_sensor, doppler_sensor, sat_range_tracker_sensor
)
from lunanav.estimation.ekf import ekf_predict, ekf_update, Qd_from_accel_white, update_sensor, update_sensor_individual_NaN_check
from lunanav.sim.quaternion import unitize_state, angle_axis_to_q, quat_apply, conj
from lunanav.plotting import plot_control_effort, plot_state_vector_combined, plot_state_vector_combined_log, plot_state_vector
from lunanav.visualization import (
    visualize_trajectory, plot_measurements, plot_attitude_relative_vertical, 
    plot_filter_confidence, obsv_verbose, plot_satellites_3d_plotly, plot_accelerometer, plot_gyroscope)
from lunanav.loaders import load_trajectory, load_ekf_result, save_ekf_result, save_trajectory, save_sim_result, load_sim_result, EKFResult, Trajectory, SimResult
from lunanav.sim.sensors import get_los_vectors
from lunanav.sim.generate import SatPosVel, make_sat_arrs, generate_env, generate_measurements


# Simulation Setup

In [ ]:
TRAJ_FILE_NAME = "piecewise_realistic"
TRAJ_FILE = f"data/trajectories/{TRAJ_FILE_NAME}.json"

In [ ]:
traj, d = load_trajectory(TRAJ_FILE)
t_arr   = traj.t
force   = jnp.array(traj.force)
torque  = jnp.array(traj.torque)
dt      = traj.dt
t_max   = traj.T
s_bar   = traj.s_bar
u_bar   = traj.u_bar
state0  = traj.state0
mass_kg = traj.mass_kg
I       = traj.I.reshape((3, 3))
nsteps  = traj.nsteps

print("Data loaded with:")
print(f"t: {t_arr.shape}, force: {force.shape}, torque: {torque.shape}, dt: {dt}, T: {t_max}, s_bar: {s_bar.shape}, u_bar: {u_bar.shape}, state0: {state0.shape}")


In [ ]:
# Reversing force and torque
state0 = np.array([
    0, 0, R_MOON + 5,      # position [m]
    0, 0, 0,               # velocity [m/s]
    1, 0, 0, 0,            # quaternion (pointed up)
    # 0, 0, 0             # angular velocity [rad/s]
    0, 0, 0             # angular velocity [rad/s]
])
# state0 = s_bar[-1]
# state0[3:6] = -state0[3:6]
# state0[2] += 10
# state0[10] = -0.00161979697307182

force = force[::-1]
torque = torque[::-1]* 0.7

# trunc_here = int(nsteps*3/4)
# force = force.at[trunc_here:].set(jnp.zeros(3))
# torque = torque.at[trunc_here:].set(jnp.zeros(3))

In [ ]:
# plot_control_effort(t_arr[:-1], force, torque)

In [ ]:
@jax.jit
def control_fn_lqr(t, state):
    del state

    index = jnp.minimum(
        jnp.array(t / dt, dtype=jnp.int32),
        len(force) - 1)
    
    force_N = force[index]  # Scalar
    torque_Nm = torque[index]  # Scalar

    return force_N, torque_Nm

@jax.jit
def control_fn_piecewise(t, state):
    """Simple control: thrust and roll torque"""
    del state
    force_N = jnp.zeros(3)
    torque_Nm = jnp.zeros(3)

    # Thrust profile
    fz = jnp.where(t < 5,   200.0,
         jnp.where(t < 30,  500.0,
         jnp.where(t < 80, 1000.0,
         jnp.where(t < 150, 300.0, 0.0))))
    force_N = jnp.array([0.0, 0.0, fz])

    # Roll torque
    tx = jnp.where(t < 6,   0.05,
         jnp.where(t < 20, -0.03,
         jnp.where(t < 40,  0.03,
         jnp.where(t < 100, -0.01, 0.0))))
    torque_Nm = jnp.array([tx, 0.0, 0.0])

    return force_N, torque_Nm

# control_fn = control_fn_lqr
control_fn = control_fn_piecewise

In [ ]:
# Simulation parameters
lander = RigidBody(
    mass_kg=mass_kg,
    I=I
)


sim = SimParams(state0, lander, dt, t_max)
print(f"Running simulation for {t_max} seconds, {nsteps} steps")
liftoff_results = run_sim(state0, nsteps, dt, control_fn, sim)
N = liftoff_results.nsteps
print(f"Simulation completed: {N} steps")
landing_results = reverse_sim_results(liftoff_results)
print("Reversed Results")

results = landing_results

In [ ]:
plot_state_vector_combined(results.t, results.states[:,0:3] - np.tile([0,0,R_MOON],(N,1)), results.states[:,3:6], results.states[:,10:13], title="State Vector")
plt.show()

In [ ]:
other_vecs = {
    "names": ["LOS1", "LOS2", "LOS3", "LOS4"],
    "vecs": get_los_vectors(),
    "colors": ['green', 'green', 'green', 'green'],
    "scale": 1e5
}

moon_offset =  np.array([0,0,R_MOON])
visualize_trajectory(results.states, results.t, dt, offset = moon_offset, title="EKF Estimated Trajectory with LOS Vectors", show_lander=False, downsample_rate=20, other_vecs=other_vecs, moon_resolution = 35).show()

# New Sensor Architecture: EKF with SensorEnvironment

## Setup

In [ ]:


# Create sensor suite with new architecture
sensor_suite = SensorSuite(sensors={
    SensorName.ACCELEROMETER: accelerometer_sensor(sigma_accel),
    SensorName.GYROSCOPE: gyroscope_sensor(sigma_gyro),
    SensorName.LASER_ALTIMETER: laser_altimeter_sensor(sigma_los),
    SensorName.LASER_VELOCITY: laser_velocity_sensor(sigma_los_vel),
    SensorName.STAR_TRACKER: star_tracker_sensor(sigma_star),
    SensorName.DOPPLER: doppler_sensor(3, sigma_doppler),
    SensorName.RANGE_TRACKER: sat_range_tracker_sensor(3, sigma_sat_range_tracker),
})

for name_enum, sensor in sensor_suite.sensors.items():
    print(f"  {name_enum.value}: {sensor.meas_dim}D, sigma = {sensor.noise_cov[0,0]**0.5:.4f}")

In [ ]:
doppler_sats: list[SatPosVel] = [
    make_sat_arrs(results.t, altitude=100e3, raan=0, aop=90, inc=90), # overhead going -x
    make_sat_arrs(results.t, altitude=100e3, raan=90, aop=94, inc=94),
    make_sat_arrs(results.t, altitude=100e3, raan=160, aop=70, inc=86),
]
r_sats = jnp.array([s.r for s in doppler_sats])
v_sats = jnp.array([s.v for s in doppler_sats])
# fig = plot_satellites_3d_plotly(doppler_sats, lander_position=results.states[:, 0:3])
# fig.show()

## Generating

In [ ]:
env_arr = generate_env(results, sim, doppler_sats)
measurements_clean, measurements_noisy = generate_measurements(results.states, env_arr, sensor_suite)
print(f"Generated {len(measurements_noisy)} measurement sets with noise")

In [ ]:
# Plot all measurements using visualization utility
fig = plot_measurements(measurements_clean, measurements_noisy, results, sensor_suite)
plt.show()

print("\nMeasurement statistics:")
print(f"Laser altitude noise: σ = {sigma_los:.1f} m")
print(f"Laser velocity noise: σ = {sigma_los_vel:.1f} m/s")
laser_alt_clean = measurements_clean[SensorName.LASER_ALTIMETER]
laser_alt_noisy = measurements_noisy[SensorName.LASER_ALTIMETER]
laser_vel_clean = measurements_clean[SensorName.LASER_VELOCITY]
laser_vel_noisy = measurements_noisy[SensorName.LASER_VELOCITY]
print(f"Max laser alt noise: {np.std(laser_alt_noisy - laser_alt_clean):.1f} m")
print(f"Max laser vel noise: {np.std(laser_vel_noisy - laser_vel_clean):.1f} m/s")

In [ ]:
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
sim_save_path = f"data/simresults/{current_time}_{TRAJ_FILE_NAME}.json"
save_sim_result(SimResult(
    trajectory_file=TRAJ_FILE,
    s_arr=results.states, force=results.force_N, torque=results.torque_Nm, t=results.t,
    dt=dt, nsteps=N, mass_kg=mass_kg, I=np.array(I),
    measurements={
        name.value: {"truth": measurements_clean[name], "noisy": measurements_noisy[name]}
        for name in sensor_suite.sensors
    },
), sim_save_path)
print("Saved sim result")


In [ ]:

sim_result = load_sim_result(sim_save_path)

# s_true   = sim_result.s_true
# t        = sim_result.t_arr
dt       = sim_result.dt
n        = sim_result.nsteps
# mass_kg  = sim_result.mass_kg
# I        = sim_result.I.reshape((3, 3))

results = SimResults(n)

results.t = sim_result.t
results.states = sim_result.s_arr
results.force_N = sim_result.force
results.torque_Nm = sim_result.torque

lander = RigidBody(
    mass_kg=sim_result.mass_kg,
    I=sim_result.I
)

measurements_clean = {
    SensorName(k): np.array(v["truth"]) for k, v in sim_result.measurements.items()
}
measurements_noisy = {
    SensorName(k): np.array(v["noisy"]) for k, v in sim_result.measurements.items()
}
print(f"Loaded sim result: {n} steps, dt={dt}s")

# EKF with Sequential Sensor Updates

In [ ]:
# EKF setup
mu_arr = np.zeros((n, 13))
Sigma_arr = np.zeros((n, 13, 13))

state0 = results.states[0]

# Initial state estimate (slightly off from truth)
mu_arr[0] = state0
# mu_arr[0] = state0 + np.array([1e3, -1e3, 1e3, 0, 0, 0, *angle_axis_to_q(0, [1,0,0], True), 0, 0, 0])
mu_arr[0] = state0 + np.array([1e3, -1e3, 1e3, 120, 20, 46, *angle_axis_to_q(30, [0,1,0], True), 10 * DEG_TO_RAD, -20 * DEG_TO_RAD, 30 * DEG_TO_RAD])
mu_arr[0] = unitize_state(mu_arr[0])
Sigma_arr[0] = np.eye(13)  # small initial uncertainty

# Process noise
Q_ekf = np.zeros((13, 13))
Q_ekf[0:6, 0:6] = Qd_from_accel_white(dt, sigma_accel)
Q_ekf[6:10, 6:10] = np.eye(4) * 1e-6
Q_ekf[10:13, 10:13] = np.eye(3) * sigma_gyro**2 * dt


print("EKF initialized")
print(f"Initial position error: {norm(mu_arr[0, 0:3] - state0[0:3]):.2f} m")

print(angle_axis_to_q(20, [0,1,0], True))

In [ ]:
# Plot true trajectory state vector

# Extract state components
r_true = results.states[:, 0:3]      # position
v_true = results.states[:, 3:6]      # velocity
q_true = results.states[:, 6:10]     # quaternion
w_true = results.states[:, 10:13]    # angular velocity

print(f"State vector ranges:")
print(f"  Position:  X=[{r_true[:, 0].min():.0f}, {r_true[:, 0].max():.0f}], Y=[{r_true[:, 1].min():.0f}, {r_true[:, 1].max():.0f}], Z=[{r_true[:, 2].min():.0f}, {r_true[:, 2].max():.0f}]")
print(f"  Velocity:  Vx=[{v_true[:, 0].min():.1f}, {v_true[:, 0].max():.1f}], Vy=[{v_true[:, 1].min():.1f}, {v_true[:, 1].max():.1f}], Vz=[{v_true[:, 2].min():.1f}, {v_true[:, 2].max():.1f}]")
print(f"  Quaternion norm (should be 1): min={np.linalg.norm(q_true, axis=1).min():.6f}, max={np.linalg.norm(q_true, axis=1).max():.6f}")
print(f"  Angular velocity: ωx=[{w_true[:, 0].min():.4f}, {w_true[:, 0].max():.4f}], ωy=[{w_true[:, 1].min():.4f}, {w_true[:, 1].max():.4f}], ωz=[{w_true[:, 2].min():.4f}, {w_true[:, 2].max():.4f}]")


In [ ]:
# DIAGNOSTICS: Check trajectory and measurement dimensions
print("=== TRAJECTORY DIAGNOSTICS ===")
# print(f"liftoff_results.nsteps: {liftoff_results.nsteps}")
# print(f"landing_results.nsteps: {landing_results.nsteps}")
print(f"n (current): {n}")
print(f"results.t length: {len(results.t)}")
print(f"results.states shape: {results.states.shape}")
print(f"results.force_N shape: {results.force_N.shape}")

print("\nMeasurement arrays shape:")
for sensor_name, meas_array in measurements_clean.items():
    print(f"  {sensor_name.value}: {meas_array.shape}")

print("\nEKF arrays shape:")
print(f"  mu_arr: {mu_arr.shape}")
print(f"  Sigma_arr: {Sigma_arr.shape}")

print("\nInitial state check:")
print(f"  state0 shape: {state0.shape}")
print(f"  state0: {state0}")
print(f"  mu_arr[0]: {mu_arr[0]}")

In [ ]:
sensor_frequencies = {
    SensorName.LASER_ALTIMETER: 1,
    SensorName.LASER_VELOCITY: 1,
    SensorName.STAR_TRACKER: 1,
    SensorName.DOPPLER: 1,
    SensorName.RANGE_TRACKER: 1,
}

In [ ]:
valid = True

# Run EKF with new sensor architecture using NOISY measurements
# Sequential updates: predict, then update with available sensors
for i in tqdm(range(n - 1)):
    force_B = results.force_N[i]
    torque_B = results.torque_Nm[i]
    accel_meas = force_B / lander.mass_kg
    gyro_meas = results.states[i, 10:13]
    # print(f"{accel_meas=}")
    # print(f"{gyro_meas=}")
    # print()

    accel_meas = measurements_noisy[SensorName.ACCELEROMETER][i]  # noisy
    gyro_meas = measurements_noisy[SensorName.GYROSCOPE][i]       # noisy
    # print(f"{accel_meas=}")
    # print(f"{gyro_meas=}")
    # print()
    # print(f"Before: {mu_arr[i]=}")
    mu_pred, Sigma_pred = ekf_predict(mu_arr[i], Sigma_arr[i], accel_meas, gyro_meas, Q_ekf, sim)
    mu_pred = unitize_state(mu_pred)
    # print(f"AFter: {mu_pred=}")

    env = env_arr[i]

    if jnp.any(jnp.isnan(mu_pred)):
        print(f"NaN after predict at i={i}")
        valid = False


    for sensor, freq in sensor_frequencies.items():
        if sensor in [SensorName.LASER_ALTIMETER, SensorName.LASER_VELOCITY]:
            mu_pred, Sigma_pred = update_sensor_individual_NaN_check(sensor, freq, mu_pred, Sigma_pred, env, sensor_suite, measurements_noisy, i)
        else:
            mu_pred, Sigma_pred = update_sensor(sensor, freq, mu_pred, Sigma_pred, env, sensor_suite, measurements_noisy, i)

        if jnp.any(jnp.isnan(mu_pred)):
            print(f"NaN after update for {sensor} at i={i}")
            valid = False
    mu_arr[i + 1] = mu_pred
    Sigma_arr[i + 1] = Sigma_pred

    if not valid:
        break

print("EKF completed")

In [ ]:
save_ekf_result(EKFResult(
    trajectory_file=TRAJ_FILE,
    mu_arr=mu_arr, Sigma_arr=Sigma_arr, t_arr=results.t,
    dt=dt, nsteps=n, mass_kg=mass_kg, I=np.array(I),
), "data/ekfresults/ilqr_skew_30_deg.json")
print("Saved EKF result")


In [ ]:
ekf_result = load_ekf_result("data/ekfresults/ilqr_skew_30_deg.json")

mu_arr    = ekf_result.mu_arr
Sigma_arr = ekf_result.Sigma_arr
t         = ekf_result.t_arr
dt        = ekf_result.dt
n         = ekf_result.nsteps
mass_kg   = ekf_result.mass_kg
I         = ekf_result.I.reshape((3, 3))
print(f"Loaded EKF result: {n} steps, dt={dt}s")

In [ ]:
plot_state_vector_combined(results.t, mu_arr[:,0:3] - np.tile([0,0,R_MOON],(n,1)), mu_arr[:,3:6], mu_arr[:,10:13], title="State Vector")

In [ ]:
# Position errors
pos_errors = np.linalg.norm(mu_arr[:, 0:3] - results.states[:, 0:3], axis=1)
vel_errors = np.linalg.norm(mu_arr[:, 3:6] - results.states[:, 3:6], axis=1)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

ax1.plot(results.t, pos_errors, label='Position Error')
ax1.set_ylabel('Position Error (m)')
ax1.set_title('EKF Estimation Error')
ax1.grid()
ax1.legend()

ax2.plot(results.t, vel_errors, label='Velocity Error')
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Velocity Error (m/s)')
ax2.grid()
ax2.legend()

plt.tight_layout()
plt.show()

print(f"Final position error: {pos_errors[-1]:.2f} m")
print(f"Final velocity error: {vel_errors[-1]:.4f} m/s")
print(f"Max position error: {np.max(pos_errors):.2f} m")

In [ ]:

# Add this to your notebook as a new cell:
fig = plot_filter_confidence(Sigma_arr, results.t)
plt.show()

# Print summary statistics
print("\n" + "="*60)
print("FILTER CONFIDENCE SUMMARY")
print("="*60)

trace_initial = np.trace(Sigma_arr[0])
trace_final = np.trace(Sigma_arr[-1])
trace_min = np.min([np.trace(Sigma_arr[i]) for i in range(len(Sigma_arr))])
trace_max = np.max([np.trace(Sigma_arr[i]) for i in range(len(Sigma_arr))])

print("\nOverall Uncertainty (Trace of Σ):")
print(f"  Initial: {trace_initial:.4e}")
print(f"  Final:   {trace_final:.4e}")
print(f"  Min:     {trace_min:.4e}")
print(f"  Max:     {trace_max:.4e}")

# Component-wise final uncertainties
print("\nFinal Standard Deviations by Component:")
print(f"  Position:          {np.linalg.norm(np.sqrt(np.diag(Sigma_arr[-1, 0:3, 0:3]))):.2f} m")
print(f"  Velocity:          {np.linalg.norm(np.sqrt(np.diag(Sigma_arr[-1, 3:6, 3:6]))):.4f} m/s")
print(f"  Attitude:          {np.linalg.norm(np.sqrt(np.diag(Sigma_arr[-1, 6:10, 6:10]))):.6f}")
print(f"  Angular Velocity:  {np.linalg.norm(np.sqrt(np.diag(Sigma_arr[-1, 10:13, 10:13]))):.6f} rad/s")


# Convergence metric
if trace_final < trace_initial:
    improvement = (trace_initial - trace_final) / trace_initial * 100
    print(f"\n✓ Filter CONVERGED: {improvement:.1f}% reduction in uncertainty")
else:
    increase = (trace_final - trace_initial) / trace_initial * 100
    print(f"\n✗ Filter DIVERGED: {increase:.1f}% increase in uncertainty")

In [ ]:
from lunanav.visualization import plot_filter_uncertainty_diag

plot_filter_uncertainty_diag(Sigma_arr, results.t, figsize=(15, 10), use_variance=False)
plt.show()

In [ ]:
# Detailed component-wise error analysis to diagnose divergence
pos_error_x = mu_arr[:, 0] - results.states[:, 0]
pos_error_y = mu_arr[:, 1] - results.states[:, 1]
pos_error_z = mu_arr[:, 2] - results.states[:, 2]

vel_error_x = mu_arr[:, 3] - results.states[:, 3]
vel_error_y = mu_arr[:, 4] - results.states[:, 4]
vel_error_z = mu_arr[:, 5] - results.states[:, 5]

fig, axes = plt.subplots(3, 2, figsize=(15, 11))

# Position components
axes[0, 0].plot(results.t, pos_error_x, 'r-', label='X error', linewidth=1)
axes[0, 0].plot(results.t, pos_error_y, 'g-', label='Y error', linewidth=1)
axes[0, 0].plot(results.t, pos_error_z, 'b-', label='Z error', linewidth=1)
axes[0, 0].set_ylabel('Position Error (m)')
axes[0, 0].set_title('Position Error - Component-wise')
axes[0, 0].grid(alpha=0.3)
axes[0, 0].legend()

axes[0, 1].semilogy(results.t, np.abs(pos_error_x) + 1, 'r-', label='|X error|', linewidth=1)
axes[0, 1].semilogy(results.t, np.abs(pos_error_y) + 1, 'g-', label='|Y error|', linewidth=1)
axes[0, 1].semilogy(results.t, np.abs(pos_error_z) + 1, 'b-', label='|Z error|', linewidth=1)
axes[0, 1].set_ylabel('Position Error (m, log scale)')
axes[0, 1].set_title('Position Error - Component-wise')
axes[0, 1].grid(alpha=0.3, which='both')
axes[0, 1].legend()

# Velocity components
axes[1, 1].semilogy(results.t, abs(vel_error_x), 'r-', label='VX error', linewidth=1)
axes[1, 1].semilogy(results.t, abs(vel_error_y), 'g-', label='VY error', linewidth=1)
axes[1, 1].semilogy(results.t, abs(vel_error_z), 'b-', label='VZ error', linewidth=1)
axes[1, 1].set_ylabel('Velocity Error (m/s, log scale)')
axes[1, 1].set_title('Velocity Error - Component-wise')
axes[1, 1].grid(alpha=0.3)
axes[1, 1].legend()

axes[1, 0].plot(results.t, np.linalg.norm(mu_arr[:, 3:6] - results.states[:, 3:6], axis=1), 'k-', linewidth=2)
axes[1, 0].set_ylabel('Velocity Error Norm (m/s)')
axes[1, 0].set_title('Total Velocity Error')
axes[1, 0].grid(alpha=0.3)

# Altitude comparison and error
altitude_true = results.states[:, 2]
altitude_est = mu_arr[:, 2]
axes[2, 0].plot(results.t, altitude_true, 'k-', label='True altitude', linewidth=2)
axes[2, 0].plot(results.t, altitude_est, 'r--', label='Estimated altitude', linewidth=2)
axes[2, 0].set_ylabel('Altitude (m)')
axes[2, 0].set_title('Altitude: Truth vs Estimate')
axes[2, 0].grid(alpha=0.3)
axes[2, 0].legend()

axes[2, 1].plot(results.t, altitude_est - altitude_true, 'r-', linewidth=2, label='Altitude error')
axes[2, 1].set_ylabel('Altitude Error (m)')
axes[2, 1].set_xlabel('Time (s)')
axes[2, 1].set_title('Altitude Error Over Time')
axes[2, 1].grid(alpha=0.3)
axes[2, 1].legend()

plt.tight_layout()
plt.show()

# Detailed statistics
print("\n" + "="*50)
print("DIVERGENCE ANALYSIS")
print("="*50)
print(f"\nPosition Error Statistics:")
print(f"  Initial: {np.linalg.norm(mu_arr[0, 0:3] - results.states[0, 0:3]):.2f} m")
print(f"  Final:   {np.linalg.norm(mu_arr[-1, 0:3] - results.states[-1, 0:3]):.2f} m")
print(f"  Max:     {np.max(np.linalg.norm(mu_arr[:, 0:3] - results.states[:, 0:3], axis=1)):.2f} m")
print(f"  Mean:    {np.mean(np.linalg.norm(mu_arr[:, 0:3] - results.states[:, 0:3], axis=1)):.2f} m")

print(f"\nVelocity Error Statistics:")
print(f"  Initial: {np.linalg.norm(mu_arr[0, 3:6] - results.states[0, 3:6]):.4f} m/s")
print(f"  Final:   {np.linalg.norm(mu_arr[-1, 3:6] - results.states[-1, 3:6]):.4f} m/s")
print(f"  Max:     {np.max(np.linalg.norm(mu_arr[:, 3:6] - results.states[:, 3:6], axis=1)):.4f} m/s")
print(f"  Mean:    {np.mean(np.linalg.norm(mu_arr[:, 3:6] - results.states[:, 3:6], axis=1)):.4f} m/s")

# Find divergence onset
pos_norms = np.linalg.norm(mu_arr[:, 0:3] - results.states[:, 0:3], axis=1)
for threshold in [100, 500, 1000, 5000]:
    idx = np.argmax(pos_norms > threshold)
    if idx > 0:
        print(f"\nError exceeds {threshold}m at step {idx}, t={results.t[idx]:.1f}s, error={pos_norms[idx]:.1f}m")
        break

print(f"\nTrajectory length: {n} steps ({results.t[-1]:.1f}s)")
print(f"Number of measurement updates: {len([i for i in range(n) if i % 10 == 0])} (every 10 steps)")

In [ ]:
# plot_state_vector(
#     results.t, 
#     mu_arr[:, 0:3], 
#     mu_arr[:, 3:6], 
#     mu_arr[:, 10:13],
#     figsize=(14, 10)
# )

# # State estimates vs truth
# plot_state_vector(
#     results.t, 
#     mu_arr[:, 0:3] - results.states[:, 0:3], 
#     mu_arr[:, 3:6] - results.states[:, 3:6], 
#     mu_arr[:, 10:13] - results.states[:, 10:13],
#     figsize=(14, 10)
# )

plot_state_vector_combined_log(results.t[-0:], mu_arr[-0:, 0:3] - results.states[-0:, 0:3], mu_arr[-0:, 3:6] - results.states[-0:, 3:6], mu_arr[-0:, 10:13] - results.states[-0:, 10:13], title="State Vector Error")
# plot_state_vector_combined_log(results.t, mu_arr[:, 0:3] - results.states[:, 0:3], mu_arr[:, 3:6] - results.states[:, 3:6], mu_arr[:, 10:13] - results.states[:, 10:13], title="State Vector Error")

In [ ]:
# Quaternion error
q_error = np.linalg.norm(mu_arr[:, 6:10] - results.states[:, 6:10], axis=1)

plt.figure(figsize=(10, 5))
plt.semilogy(results.t, abs(q_error))
plt.xlabel('Time (s)')
plt.ylabel('Quaternion Error')
plt.title('Attitude Estimation Error')
plt.grid()
plt.show()

In [ ]:
# COMPREHENSIVE STATISTICS FOR ASSESSMENT
print("\n" + "="*80)
print("COMPREHENSIVE STATISTICS & FINDINGS")
print("="*80)

# ===== TRAJECTORY STATISTICS =====
print("\n### TRAJECTORY STATISTICS ###")
print(f"Duration: {results.t[0]:.1f} to {results.t[-1]:.1f} s ({results.t[-1] - results.t[0]:.1f} s total)")
print(f"Timestep: {dt} s, Total steps: {n}")

r_truth = results.states[:, 0:3]
v_truth = results.states[:, 3:6]
print(f"\nPosition (inertial frame):")
print(f"  X: {r_truth[:, 0].min():.0f} to {r_truth[:, 0].max():.0f} m")
print(f"  Y: {r_truth[:, 1].min():.0f} to {r_truth[:, 1].max():.0f} m")
print(f"  Z: {r_truth[:, 2].min():.0f} to {r_truth[:, 2].max():.0f} m")

# Altitude above moon
altitude = r_truth[:, 2] - R_MOON
print(f"\nAltitude above moon surface:")
print(f"  Range: {altitude.min():.0f} to {altitude.max():.0f} m")
print(f"  Rate of descent: {(altitude[0] - altitude[-1]) / (results.t[-1] - results.t[0]):.1f} m/s")

print(f"\nVelocity:")
speed = np.linalg.norm(v_truth, axis=1)
print(f"  Speed: {speed.min():.1f} to {speed.max():.1f} m/s")
print(f"  Vertical velocity: {v_truth[:, 2].min():.1f} to {v_truth[:, 2].max():.1f} m/s")

# ===== MEASUREMENT STATISTICS =====
print("\n### MEASUREMENT STATISTICS ###")
print(f"Sensors used: {list(measurements_clean.keys())}")
for sensor, meas in measurements_clean.items():
    print(f"\n{sensor.name}:")
    print(f"  Shape: {meas.shape}")
    print(f"  Range: {meas.min():.2e} to {meas.max():.2e}")
    if meas.shape[1] > 1:
        for dim in range(min(3, meas.shape[1])):
            print(f"    Dim {dim}: {meas[:, dim].min():.2e} to {meas[:, dim].max():.2e}")

# ===== EKF PERFORMANCE =====
print("\n### EKF PERFORMANCE ###")
pos_err = np.linalg.norm(mu_arr[:, 0:3] - results.states[:, 0:3], axis=1)
vel_err = np.linalg.norm(mu_arr[:, 3:6] - results.states[:, 3:6], axis=1)
q_err = np.linalg.norm(mu_arr[:, 6:10] - results.states[:, 6:10], axis=1)

print(f"\nPosition error (3D):")
print(f"  Initial: {pos_err[0]:.1f} m")
print(f"  Final: {pos_err[-1]:.1f} m")
print(f"  Mean: {pos_err.mean():.1f} m")
print(f"  Max: {pos_err.max():.1f} m")
print(f"  Std: {pos_err.std():.1f} m")
print(f"  Trend: {'Diverging' if pos_err[-1] > pos_err[0] else 'Converging'}")

print(f"\nVelocity error (3D):")
print(f"  Initial: {vel_err[0]:.3f} m/s")
print(f"  Final: {vel_err[-1]:.3f} m/s")
print(f"  Mean: {vel_err.mean():.3f} m/s")
print(f"  Max: {vel_err.max():.3f} m/s")

print(f"\nAttitude error (quaternion):")
print(f"  Initial: {q_err[0]:.4f}")
print(f"  Final: {q_err[-1]:.4f}")
print(f"  Mean: {q_err.mean():.4f}")
print(f"  Max: {q_err.max():.4f}")

# ===== STATE ESTIMATE STATISTICS =====
print("\n### STATE ESTIMATE STATISTICS ###")
print(f"\nEstimated position (final):")
print(f"  {mu_arr[-1, 0:3]}")
print(f"Estimated velocity (final):")
print(f"  {mu_arr[-1, 3:6]}")
print(f"Estimated quaternion (final):")
print(f"  {mu_arr[-1, 6:10]}")

# Covariance statistics
print(f"\nEstimated uncertainty (final covariance diagonal):")
final_cov_diag = np.diag(Sigma_arr[-1])
print(f"  Position std: {np.sqrt(final_cov_diag[0:3]).mean():.1f} m")
print(f"  Velocity std: {np.sqrt(final_cov_diag[3:6]).mean():.3f} m/s")
print(f"  Attitude std: {np.sqrt(final_cov_diag[6:10]).mean():.4f}")
print(f"  Angular velocity std: {np.sqrt(final_cov_diag[10:13]).mean():.6f}")

# ===== COMPONENT-WISE ERROR =====
print(f"\n### COMPONENT-WISE ERROR (X, Y, Z) ###")
pos_error_x = mu_arr[:, 0] - results.states[:, 0]
pos_error_y = mu_arr[:, 1] - results.states[:, 1]
pos_error_z = mu_arr[:, 2] - results.states[:, 2]

print(f"\nPosition error X: {pos_error_x.mean():.1f} ± {pos_error_x.std():.1f} m (range: {pos_error_x.min():.1f} to {pos_error_x.max():.1f})")
print(f"Position error Y: {pos_error_y.mean():.1f} ± {pos_error_y.std():.1f} m (range: {pos_error_y.min():.1f} to {pos_error_y.max():.1f})")
print(f"Position error Z: {pos_error_z.mean():.1f} ± {pos_error_z.std():.1f} m (range: {pos_error_z.min():.1f} to {pos_error_z.max():.1f})")

vel_error_x = mu_arr[:, 3] - results.states[:, 3]
vel_error_y = mu_arr[:, 4] - results.states[:, 4]
vel_error_z = mu_arr[:, 5] - results.states[:, 5]

print(f"\nVelocity error X: {vel_error_x.mean():.4f} ± {vel_error_x.std():.4f} m/s")
print(f"Velocity error Y: {vel_error_y.mean():.4f} ± {vel_error_y.std():.4f} m/s")
print(f"Velocity error Z: {vel_error_z.mean():.4f} ± {vel_error_z.std():.4f} m/s")

# ===== CONVERGENCE ASSESSMENT =====
print(f"\n### CONVERGENCE ASSESSMENT ###")
early_pos_err = pos_err[:100].mean() if len(pos_err) > 100 else pos_err.mean()
late_pos_err = pos_err[-100:].mean() if len(pos_err) > 100 else pos_err.mean()
convergence_ratio = late_pos_err / early_pos_err if early_pos_err > 0 else 1.0

print(f"Early position error (first 100 steps): {early_pos_err:.1f} m")
print(f"Late position error (last 100 steps): {late_pos_err:.1f} m")
print(f"Convergence ratio: {convergence_ratio:.2f}x")
if convergence_ratio < 0.5:
    print(f"  ✓ Good convergence")
elif convergence_ratio < 1.0:
    print(f"  ~ Moderate convergence")
else:
    print(f"  ✗ DIVERGING - filter error increasing")

# ===== SUMMARY FLAGS =====
print(f"\n### KEY FINDINGS ###")
if pos_err.max() > 10000:
    print(f"⚠ WARNING: Max position error {pos_err.max():.0f} m is very large")
if convergence_ratio > 1.5:
    print(f"⚠ WARNING: Filter is diverging (error ratio: {convergence_ratio:.2f}x)")
if vel_err.max() > 10:
    print(f"⚠ WARNING: Max velocity error {vel_err.max():.2f} m/s is high")

print(f"\n" + "="*80)


In [ ]:
# # Plot attitude relative to vertical for both truth and estimate
# fig_true, tilt_true = plot_attitude_relative_vertical(results.states, results.t)
# plt.suptitle("True Trajectory - Attitude Relative to Vertical", fontsize=14, y=1.00)
# plt.show()

# fig_est, tilt_est = plot_attitude_relative_vertical(mu_arr, results.t)
# plt.suptitle("EKF Estimate - Attitude Relative to Vertical", fontsize=14, y=1.00)
# plt.show()

# # Plot both together
# fig, ax = plt.subplots(figsize=(12, 6))
# ax.plot(results.t, tilt_true, 'k-', linewidth=2, label='True')
# ax.plot(results.t, tilt_est, 'r--', linewidth=2, label='Estimated')
# ax.set_xlabel('Time (s)', fontsize=12)
# ax.set_ylabel('Tilt Angle (degrees)', fontsize=12)
# ax.set_title('Lander Tilt Angle: Truth vs EKF Estimate', fontsize=14)
# ax.grid(alpha=0.3)
# ax.legend(fontsize=11)
# plt.tight_layout()
# plt.show()

In [ ]:
# visualize_trajectory([results.states, mu_arr], results.t, dt, offset=np.array([0.0, 0.0, -R_MOON]), title="EKF Estimated Trajectory", show_lander=False, downsample_rate=20, moon_resolution=35).show()

# moon_offset =  np.tile([0,0,R_MOON,0,0,0,0,0,0,0,0,0,0], (n, 1))
moon_offset =  np.array([0,0,R_MOON])
visualize_trajectory([results.states, mu_arr], results.t, dt, offset = moon_offset, title="EKF Estimated Trajectory with LOS Vectors", show_lander=False, downsample_rate=5, moon_resolution = 35).show()

In [ ]:
# Observability analysis at different timesteps
for t_idx in [0, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 1100]:
    if t_idx < n:
        print(f"\n{'='*60}")
        print(f"Observability Analysis at t_idx={t_idx}, t={results.t[t_idx]:.1f}s")
        print(f"{'='*60}")
        obsv_verbose(
            mu_arr[t_idx],
            sensor_suite,
            results.force_N[t_idx] / lander.mass_kg,
            results.states[t_idx, 10:13],
            Q_ekf,
            sim,
            env_arr[t_idx]
        )